In [0]:
-- TRIGGER
-- Triggers in SQL are special stored procedure that automatically run in response to a specific event in a table or view.

-- TRIGGER Types
/*
    1. DDL Triggers:
        UPDATE, INSERT, DELETE
        * We are having DDL Triggers in Databricks SQL
        * Two types of DDL Triggers are
            - After
            - Instead of (runs during events)

    2. DML Triggers:
        CREATE, ALTER, DROP
        * We are not having DML Triggers in Databricks SQL
    
    3. Loggon Triggers:
        * We are not having Loggon Triggers in Databricks SQL


        SYNTAX:
      Creation =>          CREATE TRIGGER TriggerNmae on TableName
      WHEN =>              AFTER INSERT, UPDATE, DELETE
                           BEGIN
      WHAT action=>             SQL statement
                            END
*/

-- Create Table
CREATE TABLE IF NOT EXISTS workspace.default.EmployeeLogs(
    LogID BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) PRIMARY KEY,
    Emp_ID INT,
    LogMessage VARCHAR(255),
    LogDate DATE
);

-- Databricks SQL does not support CREATE TRIGGER (no automatic event-driven triggers on tables)
-- The Databricks equivalent: a stored procedure that performs the INSERT and writes the log row in the same transaction
CREATE OR REPLACE PROCEDURE workspace.default.insertEmployee(
    IN p_emp_id INT,
    IN p_name STRING,
    IN p_gender STRING,
    IN p_salary DOUBLE
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
    -- Insert the new employee row (what the original event would have done)
    INSERT INTO workspace.default.employee (emp_id, name, gender, salary)
    VALUES (p_emp_id, p_name, p_gender, p_salary);

    -- Log the insert (what the trigger body would have done)
    -- INSERTED virtual table is replaced by the procedure's input parameters
    -- '+' string concatenation is replaced by CONCAT, GETDATE() by current_date()
    INSERT INTO workspace.default.EmployeeLogs (Emp_ID, LogMessage, LogDate)
    VALUES (p_emp_id, CONCAT('New Employee Inserted: ', CAST(p_emp_id AS STRING)), current_date());
END;

-- Insert a new employee (this replaces INSERT INTO employee + trigger firing automatically)
CALL workspace.default.insertEmployee(6, 'Diana', 'F', 65000);
CALL workspace.default.insertEmployee(7, 'Maria', 'F', 75000);


-- see the log table
SELECT *
FROM workspace.default.employeelogs;
